**Workflow**

Set Google Colab Runtime type to GPU, check if GPU is being used with !nvidia-smi command.
Login to your Huggingface account and get your access token.

1. Load sample image and grad-cam functions
   * Search and select a construction vehicle dataset logging into Huggingface
   * Login to your Huggingface account and load the dataset from the Huggingface repository.
   * Display the contents of the dataset
   * Load an display an image from the dataset.
   * Save the provided pytorch-gradcam zip folder in your working directory and unzip it, change current working directory to this one.

2. Implement functions for creating Grad-cam based explanation
   * Load the class names used for training your VIT classification model and print them
   * Write a function to translate the category name to the category index.
   * Write a function to run GradCAM on an image and create a visualization.

3. Create Grad-cam visualization with model prediction
   * Load your VIT classifier model
   * Set target classes (i.e. construction vehicle classes in your model's classification output)
   * Display GradCam visualization on the model predictions
   * Print top 5 categories of the vehicle class as predicted by model

4. Implement functions for creating LIME based explanation
   * Install the python LIME library
   * Load your VIT classification model
   * Write a function to resize the input image data as accepted by your model
   * Write a predict function to run the model inference on the image and convert model predicted logits to probabilites.

5. Create and display LIME explanations
   * Load a LIME explainer
   * Run the explainer on the input image, specifying the predict function (see Step 4).
   * Create mask on image and see the areas that are encouraging the top prediction.
   * Display areas that contributes against the top prediction

In [ ]:
## Mount the google drive
import os
from google.colab import drive
drive.mount('/content/drive')

## Save the pytorch-grad-cam zip file from Setup files, unzip it on your drive and navigate to it and set it as the working directory

In [ ]:
# Change the current working directory to the specified path
os.chdir('/content/drive/My Drive/Colab Notebooks/pytorch-grad-cam')


## OR download it via git and then set it as the working directory

In [ ]:
!git clone https://github.com/jacobgil/pytorch-grad-cam.git

In [ ]:
ls

In [ ]:
pip install ttach

In [ ]:
pip install datasets

In [ ]:
# Ignore warnings to improve code readability
import warnings
warnings.filterwarnings('ignore')

# Import necessary libraries and modules
from torchvision import transforms
from datasets import load_dataset
from pytorch_grad_cam import run_dff_on_image, GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from PIL import Image
import numpy as np
import cv2
import torch
from typing import List, Callable, Optional


## Import a Huggingface dataset to use images from there for implementing XAI algorithms

In [ ]:
# Login to Huggingface
from huggingface_hub import login
login(token='Your token')

In [ ]:
## Import a Huggingface dataset

dataset = load_dataset("keremberke/excavator-detector", name="full")
image = dataset["test"]["image"][0]

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(image)

In [ ]:

img_tensor = transforms.ToTensor()(image)

""" Model wrapper to return a tensor"""
class HuggingfaceToTensorModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super(HuggingfaceToTensorModelWrapper, self).__init__()
        self.model = model

    def forward(self, x):
        return self.model(x).logits

""" Translate the category name to the category index.


"""
def category_name_to_index(model, category_name):
    name_to_index = dict((v, k) for k, v in model.config.id2label.items())
    return name_to_index[category_name]

""" Helper function to run GradCAM on an image and create a visualization.


"""
def run_grad_cam_on_image(model: torch.nn.Module,
                          target_layer: torch.nn.Module,
                          targets_for_gradcam: List[Callable],
                          reshape_transform: Optional[Callable],
                          input_tensor: torch.nn.Module=img_tensor,
                          input_image: Image=image,
                          method: Callable=GradCAM):
    with method(model=HuggingfaceToTensorModelWrapper(model),
                 target_layers=[target_layer],
                 reshape_transform=reshape_transform) as cam:

        # Replicate the tensor for each of the categories we want to create Grad-CAM for:
        repeated_tensor = input_tensor[None, :].repeat(len(targets_for_gradcam), 1, 1, 1)

        batch_results = cam(input_tensor=repeated_tensor,
                            targets=targets_for_gradcam)
        results = []
        for grayscale_cam in batch_results:
            visualization = show_cam_on_image(np.float32(input_image)/255,
                                              grayscale_cam,
                                              use_rgb=True)
            # Make it weight less in the notebook:
            visualization = cv2.resize(visualization,
                                       (visualization.shape[1]//2, visualization.shape[0]//2))
            results.append(visualization)
        return np.hstack(results)


def print_top_categories(model, img_tensor, top_k=5):
    logits = model(img_tensor.unsqueeze(0)).logits
    indices = logits.cpu()[0, :].detach().numpy().argsort()[-top_k :][::-1]
    for i in indices:
        print(f"Predicted class {i}: {model.config.id2label[i]}")

In [ ]:
# Convert the PIL Image 'image' to a NumPy array and retrieve its shape
np.array(image).shape


In [ ]:
!pip install transformers

In [ ]:
# Importing necessary modules from transformers and torchvision libraries
from transformers import ViTFeatureExtractor, ViTForImageClassification
from torchvision import transforms

# Custom reshape function for ViT Hugging Face model
def reshape_transform_vit_huggingface(x):
    activations = x[:, 1:, :]
    activations = activations.view(activations.shape[0], 12, 12, activations.shape[2])
    activations = activations.transpose(2, 3).transpose(1, 2)
    return activations

# Loading ViT model for image classification from Hugging Face
model = ViTForImageClassification.from_pretrained('google/vit-large-patch32-384')

# Defining target categories for Grad-CAM visualization
targets_for_gradcam = [
    ClassifierOutputTarget(category_name_to_index(model, "plow, plough")),
    ClassifierOutputTarget(category_name_to_index(model, "tractor")),
    ClassifierOutputTarget(category_name_to_index(model, "forklift"))
]

# Defining target layers for DFF (Deconvolutional Feature Fusion) and Grad-CAM
target_layer_dff = model.vit.layernorm
target_layer_gradcam = model.vit.encoder.layer[-2].output

# Resizing the input image and converting to a PyTorch tensor
image_resized = image.resize((384, 384))
tensor_resized = transforms.ToTensor()(image_resized)

# Displaying images with DFF and Grad-CAM visualizations along with top predicted categories
display(Image.fromarray(run_dff_on_image(model=model,
                                         target_layer=target_layer_dff,
                                         classifier=model.classifier,
                                         img_pil=image_resized,
                                         img_tensor=tensor_resized,
                                         reshape_transform=reshape_transform_vit_huggingface,
                                         n_components=4,
                                         top_k=2)))
display(Image.fromarray(run_grad_cam_on_image(model=model,
                                              target_layer=target_layer_gradcam,
                                              targets_for_gradcam=targets_for_gradcam,
                                              input_tensor=tensor_resized,
                                              input_image=image_resized,
                                              reshape_transform=reshape_transform_vit_huggingface)))
print_top_categories(model, tensor_resized)


## Implement functions for creating LIME based explanation

In [ ]:
# Importing necessary modules from PyTorch and torchvision libraries
import torch
from torchvision import models, transforms
from torch.autograd import Variable
import torch.nn.functional as F


In [ ]:
# Define a transformation to preprocess input images
def get_input_transform():
    # Normalize the image using mean and standard deviation values
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                    std=[0.229, 0.224, 0.225])
    # Compose a series of image transformations, including resizing and tensor conversion
    transf = transforms.Compose([
        transforms.Resize((384, 384)),
        # Uncomment the next line if you want to use CenterCrop(224)
        # transforms.CenterCrop(224),
        transforms.ToTensor(),
        normalize
    ])

    return transf

# Convert the input image to the format expected by the model
def get_input_tensors(img):
    transf = get_input_transform()
    # Unsqueeze converts a single image to a batch of 1
    return transf(img).unsqueeze(0)


In [ ]:
# Load the ImageNet class index information from a JSON file
import os
import json

# Initialize variables to store class index information
idx2label, cls2label, cls2idx = [], {}, {}

# Read the content of the JSON file
with open(os.path.abspath('/content/drive/My Drive/Colab Notebooks/imagenet_class_index.json'), 'r') as read_file:
    class_idx = json.load(read_file)
    # Extract class labels and indices
    idx2label = [class_idx[str(k)][1] for k in range(len(class_idx))]
    cls2label = {class_idx[str(k)][0]: class_idx[str(k)][1] for k in range(len(class_idx))}
    cls2idx = {class_idx[str(k)][0]: k for k in range(len(class_idx))}


In [ ]:
# Transform the input image to the format expected by the model
img_t = get_input_tensors(image)

# Set the model to evaluation mode
model.eval()

# Perform forward pass to obtain model predictions (logits)
logits = model(img_t)


In [ ]:
# Softmax normalization to obtain probability scores for each class
from scipy.special import softmax
with torch.no_grad():
    # Apply softmax along the specified axis (axis=1) to get class probabilities
    probs = softmax(logits[0], axis=1)
    # Convert the probabilities to a PyTorch tensor
    probs = torch.tensor(probs)
    # Get the top 5 probabilities and their corresponding class indices
    probs5 = probs.topk(5)

# Create a tuple containing probability, class index, and class label for the top 5 predictions
result_tuple = tuple((p, c, idx2label[c]) for p, c in zip(probs5[0][0].detach().numpy(), probs5[1][0].detach().numpy()))


In [ ]:
def get_pil_transform():
    # Create a PIL transform for resizing the image
    transf = transforms.Compose([
        transforms.Resize((384, 384))
    ])
    return transf

def get_preprocess_transform():
    # Create a PyTorch transform for preprocessing the image
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    transf = transforms.Compose([
        transforms.ToTensor(),
        normalize
    ])
    return transf

# Get PIL and preprocess transforms
pil_transform = get_pil_transform()
preprocess_transform = get_preprocess_transform()


In [ ]:
def batch_predict(images):
    # Set the model to evaluation mode
    model.eval()

    # Stack the preprocessed images into a batch
    batch = torch.stack(tuple(preprocess_transform(i) for i in images), dim=0)

    # Determine the device (GPU or CPU) and move the model and batch to that device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    batch = batch.to(device)

    # Perform the forward pass and calculate softmax probabilities
    logits = model(batch)
    with torch.no_grad():
        probs = softmax(logits[0], axis=1)
        probs = torch.tensor(probs)

    # Convert the probabilities to NumPy array and return
    return probs.detach().cpu().numpy()


In [ ]:
# Preprocess the image using the PIL transform
test_pil_image = pill_transf(image)

# Perform batch prediction on the preprocessed image
test_pred = batch_predict([test_pil_image])

# Extract the index of the predicted class with the highest probability
predicted_class_index = test_pred.squeeze().argmax()


In [ ]:
pip install lime

In [ ]:
# Create a LIME Image Explainer
from lime import lime_image
explainer = lime_image.LimeImageExplainer()

# Generate an explanation for the image using LIME
explanation = explainer.explain_instance(
    np.array(pill_transf(image)),  # Image in NumPy format
    batch_predict,  # Classification function
    top_labels=5,
    hide_color=0,
    num_samples=1000
)


In [ ]:
# Import the necessary function for generating image boundaries
from skimage.segmentation import mark_boundaries

# Get the image and mask for the top predicted label with positive features
temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0],
    positive_only=True,
    num_features=5,
    hide_rest=False
)

# Create an image with marked boundaries
# Display the image with marked boundaries

img_boundry1 = mark_boundaries(temp/255.0, mask)
plt.imshow(img_boundry1)


In [ ]:
temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], positive_only=False, num_features=10, hide_rest=False)
img_boundry2 = mark_boundaries(temp/255.0, mask)
plt.imshow(img_boundry2)